# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. The dataset is defined via a Croissant schema and is designed for AI-ready data standards.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata can be accessed as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` field for clarity and consistency.

Below, we list record sets and their details, including field `@id`s.

In [ ]:
# List all record sets and their @id
record_sets = []
for record_set in dataset.record_sets():
    print(f"RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name','')}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    Field @id: {field['@id']} Name: {field.get('name','')} DataType: {field.get('dataType','')}")
    record_sets.append(record_set['@id'])

if not record_sets:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis.
Each record set and field is referenced by its `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet @id: {record_set_id}")

# Show columns and preview for the first record set
if len(record_sets) > 0:
    rs0 = record_sets[0]
    print(f"Columns in RecordSet @id {rs0}:")
    print(dataframes[rs0].columns.tolist())
    display(dataframes[rs0].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the extracted data:
- Filter records by a numeric field (e.g., GAD-7 or PHQ-9 score)
- Normalize numeric fields
- Group data by a demographic attribute (e.g., gender, education)

Referencing all columns by their `@id`.

In [ ]:
# Choose a record set for EDA
record_set_id = record_sets[0] if record_sets else None

if record_set_id:
    df = dataframes[record_set_id]
    print("Available columns:")
    print(df.columns.tolist())

    # Example: Assume fields named with their @id in Croissant, e.g. 'cr:GAD7_score', 'cr:PHQ9_score', etc.
    # You may need to cross-check your schema or columns for exact @id (below are common conventions).
    numeric_fields = [col for col in df.columns if 'GAD' in col or 'PHQ' in col or 'score' in col or 'age' in col or 'psq' in col.lower()]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field @id: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 10
        try:
            threshold = float(threshold)
        except:
            threshold = 10

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Attempt to group by a demographic column
        group_fields = [col for col in df.columns if ('gender' in col.lower()) or ('education' in col.lower())]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize numeric score distributions and relationships between demographic fields.

All fields are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for numeric fields if available
if record_set_id and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet @id: {record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

    # If group_field exists, plot grouped means
    if group_fields:
        grouped = df.groupby(group_field)[numeric_field].mean()
        plt.figure(figsize=(7,4))
        grouped.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

Key findings and typical workflow steps:
- Used dataset record sets and field `@id`s for all references.
- Loaded all available record sets and previewed columns and values.
- Performed EDA by filtering, normalizing, and grouping by demographic features.
- Visualized numeric scores and their distributions.

Continue exploring the dataset by referencing additional fields and record sets using their `@id` as needed.